# Pancancer enrichment analysis step 7: Make figures from GSEApy data with weighted score type = 2

## Setup

In [1]:
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd
import operator
import IPython.display

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Step 1 output
STEP01_DIR = "step01_outputs"
STEP01_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP01_FILE_PATH = os.path.join(STEP01_DIR, STEP01_FILE)

# Step 5 outputs
STEP05_DIR = "step05_outputs"

STEP05_abs_signal_to_noise = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_abs_signal_to_noise.tsv")
STEP05_diff_of_classes = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_diff_of_classes.tsv")
STEP05_log2_ratio_of_classes = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_log2_ratio_of_classes.tsv")
STEP05_ratio_of_classes = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_ratio_of_classes.tsv")
STEP05_signal_to_noise = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_signal_to_noise.tsv")
STEP05_t_test = os.path.join(STEP05_DIR, "enrichment_gseapy_reactome_20200618_104511_10000_perms_wst2_t_test.tsv")

# Step 6 output
STEP06_DIR = "step06_outputs"

STEP06_S2N = "enrichment_gseapy_kegg_signal_to_noise_10000_perms_20200617_072701.tsv"
STEP06_S2N_PATH = os.path.join(STEP06_DIR, STEP06_S2N)

STEP06_ABS_S2N = "enrichment_gseapy_kegg_abs_signal_to_noise_10000_perms_20200617_103905.tsv"
STEP06_ABS_S2N_PATH = os.path.join(STEP06_DIR, STEP06_ABS_S2N)

MSV = 0.5 # Simplified expression value for marginally significant proteins

In [3]:
# Altair options
# alt.data_transformers.disable_max_rows()
# alt.data_transformers.enable('json')

## Function to plot top pathways across cancer types

In [4]:
def plot_top_ten(enrich_file_path, expr_file_path, xtitle, fdr=0.3):
    
    # Read in the expression data
    all_expression_data = pd.read_csv(expr_file_path, sep="\t", index_col=0)

    # Make a column where all increases are +1 and all decreases 
    # are -1, because these are ratioed abundances, so we can't 
    # compare magnitudes between genes--we can only compare whether 
    # there was a change, and whether it was positive or negative
    all_expression_data = all_expression_data.assign(simplified_change=np.nan)

    # adj p < 0.05 and change > 1 => +1
    all_expression_data.loc[
        (all_expression_data["change"] > 0) & (all_expression_data["adj_p"] < 0.05), 
        "simplified_change"
    ] = 1

    # adj p >= 0.05 and change > 1 => +0.5
    all_expression_data.loc[(all_expression_data["change"] > 0) & (all_expression_data["adj_p"] >= 0.05),
        "simplified_change"
    ] = MSV

    # change == 0 => 0
    all_expression_data.loc[
        all_expression_data["change"] == 0,
        "simplified_change"
    ] = 0

    # adj p >= 0.05 and change < 1 => -0.5
    all_expression_data.loc[(all_expression_data["change"] < 0) & (all_expression_data["adj_p"] >= 0.05), 
        "simplified_change"
    ] = -MSV

    # adj p < 0.05 and change < 1 => -1
    all_expression_data.loc[
        (all_expression_data["change"] < 0) & (all_expression_data["adj_p"] < 0.05),
        "simplified_change"
    ] = -1

    # Select just the proteins where we chose to reject the null hypothesis of no change
    expression_data = all_expression_data[all_expression_data["reject_null"]]

    # Read in the enrichment data
    enrichment_data = pd.read_csv(enrich_file_path, sep="\t")
    
    # Select enrichment data below our FDR
    enrichment_data = enrichment_data.loc[enrichment_data["fdr"] <= fdr, :]
    
    # Make a column for abs_nes
    enrichment_data = enrichment_data.assign(abs_nes=enrichment_data["nes"].abs())

    # Assign pathway ranks within each cancer type based on absolute value of NES
    enrichment_data = enrichment_data.\
        assign(cancer_rank_abs_nes=enrichment_data.groupby("cancer_type")["abs_nes"].rank(ascending=False)).\
        sort_values(by=["cancer_type", "cancer_rank_abs_nes"]).\
        reset_index(drop=True)

    # Make a table with summary info for all pathways
    enrichment_summary = enrichment_data[["Term"]].drop_duplicates(keep="first")

    pathway_times_enriched = enrichment_summary["Term"].apply(
        lambda x: enrichment_data[enrichment_data["Term"] == x].shape[0])

    avg_rank = enrichment_summary["Term"].apply(
        lambda x: enrichment_data.loc[enrichment_data["Term"] == x, "cancer_rank_abs_nes"].mean())

    enrichment_summary = enrichment_summary.\
        assign(
            pathway_times_enriched=pathway_times_enriched,
            pathway_avg_rank=avg_rank).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank"],
            ascending=[False, True]).\
        reset_index(drop=True)

    # Merge in the original enrichment data
    enrichment_data = enrichment_data.\
        merge(
            enrichment_summary,
            how="outer",
            left_on="Term",
            right_on="Term",
            validate="many_to_one"
        ).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank", "cancer_type"],
            ascending=[False, True, True]
        )

    # Select top 10 for our plot
    top_ten = enrichment_data["Term"].unique()[:10]
    sel_enrich = enrichment_data[enrichment_data["Term"].isin(top_ten)]

    # Calculate the mean expression for each pathway in each cancer type
    mean_exprs = []

    for idx in sel_enrich.index:
        genes = sel_enrich.loc[idx, "genes"].split(";")
        cancer_type = sel_enrich.loc[idx, "cancer_type"]

        genes_expr = expression_data.loc[
            expression_data["protein_str"].isin(genes),
            "simplified_change"
        ].mean()

        mean_exprs.append(genes_expr)

    sel_enrich = sel_enrich.assign(mean_expr=mean_exprs)

    sel_enrich = sel_enrich.assign(
        rank_size=1 / sel_enrich["cancer_rank_abs_nes"],
        avg_rank_size=1 / sel_enrich["pathway_avg_rank"])

    individual = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "Term:N",
            sort=sel_enrich["Term"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="",
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            axis=alt.Axis(
                title="Cancer type",
                titleFontSize=12
            ),
        ),
        size=alt.Size(
            "rank_size:Q",
            legend=None
        ),
        color=alt.Color(
            "mean_expr:Q",
            scale=alt.Scale(
                scheme="blueorange",
                domain=[-1, 1]
            ),
            legend=alt.Legend(
                title="Pathway tumor expression"
            )
        )
    ).properties(
        width=400,
        height=300
    )

    aggregate = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "pathway_avg_rank:N",
            sort=sel_enrich["pathway_avg_rank"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="Overall rank of pathway",
                titleFontSize=12
            )
        ),
        size=alt.Size(
            "avg_rank_size:Q",
            legend=None
        ),
    ).properties(
        width=400
    )

    full_plot = alt.vconcat(
        aggregate, individual
    ).properties(
        title=xtitle
    )
    
    return full_plot, enrichment_data, all_expression_data, enrichment_summary

In [5]:
plot_top_ten(STEP05_abs_signal_to_noise, STEP01_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Lysosphingolipid and LPA receptors,1,1.0
1,O-linked glycosylation of mucins,1,1.0
2,RNA Polymerase III Abortive And Retractive Ini...,1,1.5
3,RNA Polymerase III Transcription,1,1.5
4,Defective SLC7A7 causes lysinuric protein into...,1,2.0
5,Complex I biogenesis,1,2.0
6,RNA Polymerase III Transcription Initiation,1,3.0
7,RNA Polymerase III Transcription Initiation Fr...,1,4.0
8,"Defective POMGNT1 causes MDDGA3, MDDGB3 and MD...",1,4.0
9,"Defective POMT1 causes MDDGA1, MDDGB1 and MDDGC1",1,4.0


In [6]:
plot_top_ten(STEP05_diff_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,HIV Life Cycle,2,9.0
1,Transport of Mature Transcript to Cytoplasm,2,13.5
2,Late Phase of HIV Life Cycle,2,14.0
3,Transport of Mature mRNAs Derived from Intronl...,2,14.0
4,rRNA processing,2,15.0
5,Transport of Mature mRNA Derived from an Intro...,2,16.0
6,Influenza Infection,2,17.0
7,rRNA processing in the nucleus and cytosol,2,19.0
8,Major pathway of rRNA processing in the nucleo...,2,19.5
9,Translation,2,21.0


In [7]:
plot_top_ten(STEP05_log2_ratio_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank


In [8]:
plot_top_ten(STEP05_ratio_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Synthesis of 12-eicosatetraenoic acid derivatives,4,49.00
1,Trafficking of myristoylated proteins to the c...,2,5.50
2,Metal sequestration by antimicrobial proteins,2,10.50
3,Inhibition of Signaling by Overexpressed EGFR,2,11.00
4,Signaling by Overexpressed Wild-Type EGFR in C...,2,11.00
5,NF-kB is activated and signals survival,2,13.50
6,ARMS-mediated activation,2,14.50
7,Muscle contraction,2,19.50
8,Toxicity of botulinum toxin type D (BoNT/D),2,28.25
9,Toxicity of botulinum toxin type F (BoNT/F),2,28.25


In [9]:
plot_top_ten(STEP05_signal_to_noise, STEP01_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",8,112.125000
1,Negative regulation of the PI3K/AKT network,8,122.750000
2,Paradoxical activation of RAF signaling by kin...,8,153.500000
3,Signaling by RAS mutants,8,153.500000
4,Signaling by moderate kinase activity BRAF mut...,8,153.500000
5,Signaling downstream of RAS mutants,8,153.500000
6,GPCR downstream signalling,7,24.285714
7,G alpha (i) signalling events,7,24.285714
8,Signaling by GPCR,7,26.285714
9,G alpha (q) signalling events,7,40.571429


In [10]:
plot_top_ten(STEP05_t_test, STEP01_FILE_PATH, "Reactome data - proteins ranked by t_test")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",8,128.125000
1,Paradoxical activation of RAF signaling by kin...,8,143.000000
2,Signaling by RAS mutants,8,143.000000
3,Signaling by moderate kinase activity BRAF mut...,8,143.000000
4,Signaling downstream of RAS mutants,8,143.000000
5,Negative regulation of the PI3K/AKT network,8,143.625000
6,GPCR downstream signalling,7,22.428571
7,Signaling by GPCR,7,23.571429
8,G alpha (i) signalling events,7,24.000000
9,Neuronal System,7,34.571429


In [11]:
df = pd.read_csv(STEP05_log2_ratio_of_classes, sep="\t", index_col=0)

In [12]:
df["fdr"].min()

0.7260497213801732

In [13]:
alt.vconcat(
    plot_top_ten(STEP05_abs_signal_to_noise, STEP01_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[0],
    plot_top_ten(STEP05_diff_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[0],
    plot_top_ten(STEP05_log2_ratio_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[0],
    plot_top_ten(STEP05_ratio_of_classes, STEP01_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[0],
    plot_top_ten(STEP05_signal_to_noise, STEP01_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[0],
    plot_top_ten(STEP05_t_test, STEP01_FILE_PATH, "Reactome data - proteins ranked by t_test")[0],
    
    plot_top_ten(STEP06_ABS_S2N_PATH, STEP01_FILE_PATH, "KEGG data - proteins ranked by |S2N|")[0],
    plot_top_ten(STEP06_S2N_PATH, STEP01_FILE_PATH, "KEGG data - proteins ranked by S2N")[0],
).configure_axis(
    grid=True
).configure_title(
    fontSize=16,
    anchor="end",
    offset=20
).resolve_scale(
    size="independent"
)

alt.VConcatChart(...)

In [14]:
def find_neutrophil_degranulation(fp):
    enrich = plot_top_ten(fp, STEP01_FILE_PATH, "Reactome data - proteins ranked by |S2N|", fdr=1)[1]

    # enrich = enrich[enrich["pathway_times_enriched"] == 8]
    ranks = enrich[["Term", "pathway_avg_rank"]].\
        set_index("Term").\
        drop_duplicates(keep="first").\
        rank()["pathway_avg_rank"].\
        rename("pathway_overall_rank")

    enrich = enrich.merge(
        right=ranks,
        left_on="Term",
        right_on="Term",
        validate="many_to_one")

    return enrich[enrich["Term"] == "Neutrophil degranulation"]

In [15]:
find_neutrophil_degranulation(STEP05_abs_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
9688,Neutrophil degranulation,0.608472,1.293213,0.000442,0.619736,479,427,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,ccrcc,1.293213,931.0,8,1389.125,1390.0
9689,Neutrophil degranulation,0.516131,1.072644,0.204196,0.713143,479,405,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,colon,1.072644,1501.0,8,1389.125,1390.0
9690,Neutrophil degranulation,0.491253,1.024331,0.371233,0.788814,479,427,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,endometrial,1.024331,1593.0,8,1389.125,1390.0
9691,Neutrophil degranulation,0.510039,1.068778,0.283258,0.774740,479,432,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,gbm,1.068778,1494.0,8,1389.125,1390.0
9692,Neutrophil degranulation,0.595267,1.224650,0.010643,0.483641,479,448,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,hnscc,1.224650,1267.0,8,1389.125,1390.0
9693,Neutrophil degranulation,0.507608,1.064419,0.250224,0.758287,479,423,SPTAN1;EEF2;VAT1;CD93;CD55;CAT;SERPINB6;PECAM1...,SPTAN1;EEF2;VAT1;CD93;CD55;CAT;SERPINB6;PECAM1...,lscc,1.064419,1538.0,8,1389.125,1390.0
9694,Neutrophil degranulation,0.522285,1.083354,0.191874,0.698388,479,409,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,luad,1.083354,1530.0,8,1389.125,1390.0
9695,Neutrophil degranulation,0.555669,1.142007,0.030455,0.686213,479,418,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,ovarian,1.142007,1259.0,8,1389.125,1390.0


In [16]:
find_neutrophil_degranulation(STEP05_diff_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
10264,Neutrophil degranulation,0.485985,2.015028,0.429571,0.723072,479,427,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,ccrcc,2.015028,1580.0,8,1334.75,1356.0
10265,Neutrophil degranulation,-0.400804,-1.679048,0.711466,0.815261,479,405,S100P;S100A9;S100A8;S100A11;S100A12;CAMP;LCN2;...,VAT1;FGL2;STOM;SLC44A2;PIGR;IQGAP2;PTX3;RAB5B;...,colon,1.679048,1854.0,8,1334.75,1356.0
10266,Neutrophil degranulation,0.445434,1.763212,0.626374,0.892793,479,427,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,endometrial,1.763212,1667.0,8,1334.75,1356.0
10267,Neutrophil degranulation,0.370392,1.412316,0.831283,0.934764,479,432,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,gbm,1.412316,2003.0,8,1334.75,1356.0
10268,Neutrophil degranulation,0.330453,1.288301,1.000000,1.000000,479,448,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,hnscc,1.288301,1736.0,8,1334.75,1356.0
10269,Neutrophil degranulation,-0.548766,-2.340875,0.337121,0.637339,479,423,DSP;PKP1;PLAU;SERPINB3;CKAP4;JUP;TNFAIP6;S100A...,C5AR1;DPP7;CPPED1;PDXK;CD63;BRI3;COMMD9;FOLR3;...,lscc,2.340875,995.0,8,1334.75,1356.0
10270,Neutrophil degranulation,-0.634523,-2.579441,0.171904,0.616752,479,409,DSP;PLAU;TXNDC5;CKAP4;MLEC;NEU1;DNAJC3;FUCA1;H...,CEACAM8;STK10;SVIP;DEFA4;NAPRT;DNASE1L1;C5AR1;...,luad,2.579441,476.0,8,1334.75,1356.0
10271,Neutrophil degranulation,-0.705165,-2.854049,0.033929,0.327233,479,418,NPC2;PKM;HSP90AB1;ILF2;MIF;SRP14;HSP90AA1;CD47...,LILRB3;HGSNAT;GYG1;RNASE2;GSN;SPTAN1;SIRPA;SNA...,ovarian,2.854049,367.0,8,1334.75,1356.0


In [17]:
find_neutrophil_degranulation(STEP05_log2_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
1848,Neutrophil degranulation,0.426247,1.068288,0.976017,1.0,479,448,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,hnscc,1.068288,2040.0,1,2040.0,1849.0


In [18]:
find_neutrophil_degranulation(STEP05_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
11501,Neutrophil degranulation,0.621176,1.486440,0.798333,0.648018,479,427,PPIE;PSMD6;CHI3L1;RHOA;PSMD2;SERPINB10;DEFA4;R...,PPIE;PSMD6;CHI3L1;RHOA,ccrcc,1.486440,1865.0,7,1528.571429,1760.0
11502,Neutrophil degranulation,0.696961,1.746600,0.705292,0.964935,479,427,TNFAIP6;CD300A;CD55;ALDH3B1;UNC13D;PGM1;LAMTOR...,TNFAIP6;CD300A;CD55,endometrial,1.746600,1272.0,7,1528.571429,1760.0
11503,Neutrophil degranulation,-0.786321,-1.810411,0.728164,1.000000,479,432,CYB5R3;CEACAM8;PLAUR;GM2A;ORMDL3;PSMB7;ATAD3B;...,PRTN3;CD177;VPS35L;VCP;PIGR;MPO;PSMD6;PGLYRP1;...,gbm,1.810411,1253.0,7,1528.571429,1760.0
11504,Neutrophil degranulation,0.140088,0.753681,0.535675,1.000000,479,448,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,hnscc,0.753681,2135.0,7,1528.571429,1760.0
11505,Neutrophil degranulation,0.702175,1.752566,0.658772,1.000000,479,423,MMP8;RETN;MAN2B1;GM2A;PADI2;FABP5;CD300A;KCMF1...,MMP8;RETN;MAN2B1;GM2A;PADI2;FABP5;CD300A;KCMF1...,lscc,1.752566,1447.0,7,1528.571429,1760.0
11506,Neutrophil degranulation,-0.669114,-1.724073,0.719523,1.000000,479,409,PSMB7;GHDC;RAB6A;HLA-B;TMC6;SNAP29;CHIT1;EPX;C...,CAMP;NHLRC3;VAPA;CEACAM6;ADGRE5;ADGRE3;S100A9,luad,1.724073,1437.0,7,1528.571429,1760.0
11507,Neutrophil degranulation,0.737149,1.846987,0.633243,0.717695,479,418,DIAPH1;HSPA6;TUBB;GSDMD;LRMP;ALOX5;KRT1;ARSB;R...,DIAPH1;HSPA6;TUBB;GSDMD;LRMP;ALOX5;KRT1;ARSB;R...,ovarian,1.846987,1291.0,7,1528.571429,1760.0


In [19]:
find_neutrophil_degranulation(STEP05_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
9536,Neutrophil degranulation,0.475020,2.675099,0.074301,0.280368,479,427,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;LPCAT1;CYBB;ALD...,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;LPCAT1;CYBB;ALD...,ccrcc,2.675099,673.0,8,1268.0,1226.0
9537,Neutrophil degranulation,-0.287120,-1.611639,0.746729,0.834970,479,405,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,RAC1;PPBP;IST1;STOM;ORM2;SLC44A2;RNASE2;SLPI;A...,colon,1.611639,1886.0,8,1268.0,1226.0
9538,Neutrophil degranulation,0.395978,2.034721,0.399107,0.883656,479,427,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;NBEAL2;CTS...,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;NBEAL2;CTS...,endometrial,2.034721,1325.0,8,1268.0,1226.0
9539,Neutrophil degranulation,-0.349314,-1.965900,0.575150,0.729371,479,432,IDH1;PGM2;STK10;IQGAP2;EEF1A1;PYGL;SERPINB6;TY...,ATP11B;ARMC8;RAB37;UBR4;PYGB;HSP90AA1;CD59;ATP...,gbm,1.965900,1600.0,8,1268.0,1226.0
9540,Neutrophil degranulation,-0.349861,-2.001202,0.469479,0.670004,479,448,CNN2;PLAU;IGF2R;LPCAT1;FCER1G;HSP90AB1;ITGAX;M...,CPNE3;SDCBP;CYB5R3;PPBP;AP2A2;NDUFC2;AGL;VCL;Q...,hnscc,2.001202,1693.0,8,1268.0,1226.0
9541,Neutrophil degranulation,-0.487472,-2.515683,0.229417,0.404370,479,423,EEF2;CKAP4;DDX3X;DSP;HSP90AB1;PLAU;PA2G4;ILF2;...,RAB4B;MVP;NBEAL2;TYROBP;CTSD;GMFG;CPPED1;TOM1;...,lscc,2.515683,934.0,8,1268.0,1226.0
9542,Neutrophil degranulation,-0.480369,-2.495912,0.181244,0.406434,479,409,COPB1;EEF2;SRP14;HSP90AB1;PSMD3;PSMD6;DNAJC3;P...,ATP6V1D;PRTN3;SVIP;OLR1;CFP;CD47;NAPRT;CTSH;SI...,luad,2.495912,930.0,8,1268.0,1226.0
9543,Neutrophil degranulation,-0.406022,-2.417247,0.265247,0.532834,479,418,HSP90AB1;HSP90AA1;ILF2;COPB1;EEF1A1;EEF2;NPC2;...,PLD1;MAPK1;ADGRE5;ACTR1B;CD14;SNAP25;SERPINB3;...,ovarian,2.417247,1103.0,8,1268.0,1226.0


In [20]:
find_neutrophil_degranulation(STEP05_t_test)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
9224,Neutrophil degranulation,0.473255,2.657259,0.078466,0.284596,479,427,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,ccrcc,2.657259,713.0,8,1264.75,1186.0
9225,Neutrophil degranulation,-0.290066,-1.625156,0.733894,0.832815,479,405,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,RAC1;PPBP;STOM;ORM2;SLC44A2;RNASE2;SLPI;CST3;A...,colon,1.625156,1871.0,8,1264.75,1186.0
9226,Neutrophil degranulation,0.388068,1.991558,0.423592,0.900531,479,427,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,endometrial,1.991558,1377.0,8,1264.75,1186.0
9227,Neutrophil degranulation,-0.324534,-1.801663,0.652131,0.808884,479,432,IDH1;PGM2;EEF1A1;IQGAP2;APAF1;STK10;CPNE3;TYRO...,UBR4;SLC2A3;ATP11B;HSP90AA1;PYGB;RAB37;CCT8;SN...,gbm,1.801663,1704.0,8,1264.75,1186.0
9228,Neutrophil degranulation,-0.358621,-2.063110,0.438233,0.638147,479,448,CNN2;PLAU;LPCAT1;IGF2R;FCER1G;HSP90AB1;MNDA;IT...,CPNE3;SDCBP;NDUFC2;CYB5R3;LAMTOR2;AGL;PPBP;AP2...,hnscc,2.063110,1619.0,8,1264.75,1186.0
9229,Neutrophil degranulation,-0.497411,-2.565932,0.215541,0.372602,479,423,EEF2;CKAP4;DDX3X;DSP;HSP90AB1;ILF2;PLAU;PA2G4;...,PYGB;PDXK;ATP6V1D;GHDC;ADAM10;SELL;PNP;RAB14;B...,lscc,2.565932,863.0,8,1264.75,1186.0
9230,Neutrophil degranulation,-0.489549,-2.545198,0.161698,0.365193,479,409,COPB1;SRP14;EEF2;PSMD3;HSP90AB1;PSMD6;PDAP1;DN...,VNN1;OSTF1;PGLYRP1;LTA4H;CREG1;NAPRT;PRTN3;OLR...,luad,2.545198,861.0,8,1264.75,1186.0
9231,Neutrophil degranulation,-0.406158,-2.420571,0.266990,0.532676,479,418,HSP90AB1;HSP90AA1;ILF2;COPB1;EEF1A1;EEF2;PKM;X...,HEBP2;SLPI;S100A9;HK3;DOK3;MNDA;QSOX1;ATP8A1;D...,ovarian,2.420571,1110.0,8,1264.75,1186.0


## Figure 2: Expression of proteins in the neutrophil degranulation pathway

In [ ]:
# Get the ID the top expessed pathway
neutro_pathway_ids = enrichment_data.\
    loc[enrichment_data["name"] == "Neutrophil degranulation", "pathway_id"].\
    unique()

if len(neutro_pathway_ids) != 1:
    raise ValueError("Multiple pathways found.")
    
neutro_pathway_id = neutro_pathway_ids[0]

### A: With proteins in different orders for the up and down categories

In [ ]:
def expr_heatmap(expr_data, enrich_data, pathway_id, up_or_down, max_p, x_title=True):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    x_title (bool): Whether to include an x axis title.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
           ["cancer_type", "mean_expr"]]

    # Select data for the specified cancer types
    selected_expr_data = pathway_expr[pathway_expr["cancer_type"].isin(sel_cancers["cancer_type"])]

    # Sort proteins by mean expression across selected cancer types
    prot_order = selected_expr_data.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    cancer_order = sel_cancers.\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, "cancer_type"]
    
    # Map simplified_change to legend values
    selected_expr_data = selected_expr_data.assign(
        tumor_up_down=selected_expr_data["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(selected_expr_data).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (different order in each facet)" if x_title else ""
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order.values,
            axis=alt.Axis(
                title=f"Pathway {up_or_down} in tumor"
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
    ).properties(
        width=700,
        height=175
    )
    
    return plot

In [ ]:
# Plot
alt.vconcat(
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "up", max_p=0.2, x_title=False),
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "down", max_p=0.2)
)

### B: With same sorting for both

In [ ]:
def expr_heatmap_alt_sort(expr_data, enrich_data, pathway_id, max_p):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Sort proteins by mean expression across all cancer types
    prot_order = pathway_expr.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    sel_cancers = enrich_data.\
        loc[enrich_data["pathway_id"] == neutro_pathway_id, ["cancer_type", "mean_expr"]]
        
    sel_cancers = sel_cancers.\
        assign(up_or_down=np.where(sel_cancers["mean_expr"] > 0, "Pathway up in tumor", "Pathway down in tumor")).\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, ["cancer_type", "up_or_down"]]
    
    cancer_order = sel_cancers["cancer_type"].values
    
    pathway_expr = pathway_expr.merge(
        right=sel_cancers,
        how="inner",
        left_on="cancer_type",
        right_on="cancer_type",
        validate="many_to_one"
    )
    
    # Map simplified_change to legend values
    pathway_expr = pathway_expr.assign(
        tumor_up_down=pathway_expr["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(pathway_expr).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (same order in both facets)"
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order,
            axis=alt.Axis(
                title=""
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
        row=alt.Row(
            "up_or_down:N",
            sort="descending",
            title="",
            header=alt.Header(
                labelFontStyle="bold"
            )
        )
    ).properties(
        width=600,
        height=150
    ).resolve_scale(
        y="independent" # This makes it so each facet (up or down) only has the cancers that are in that category
    )
    
    return plot

In [ ]:
# Plot
expr_heatmap_alt_sort(all_expression_data, enrichment_data, neutro_pathway_id, max_p=0.2)

## Figure 3: Pathway overlay for neutrophil degranulation

In [ ]:
def pathway_overlay_wrapper(pathway_id, exp_data, enrich_data, up_or_down):
    
    prots = ut.search_reactome_proteins_in_pathways(pathway_id)
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
            "cancer_type"]
    
    # Select the desired expression data and average the simplified change across cancer types
    sel_exp = exp_data.\
        loc[
            exp_data["protein_str"].isin(prots["member"]) & 
            exp_data["cancer_type"].isin(sel_cancers),
            ["protein_str", "simplified_change"]
        ].\
        groupby("protein_str").\
        agg(np.mean).\
        rename(columns={"simplified_change": f"{up_or_down}_simp_change"})
    
    img_name = f"{TIME_START}_{pathway_id}_{up_or_down}.png"
    img_path = os.path.join(STEP07_DIR, img_name)
    
    token, _ = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=sel_exp, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    _, img_path = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
        export_path=img_path,
        image_format="png",
        display_col_idx=None,
        diagram_colors="Modern",
        overlay_colors="Standard",
        quality=10
    )

    _, url = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
    )

    return img_path, url

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

up_image_path, up_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "up")

In [ ]:
IPython.display.HTML(f'<a href={up_url}>{up_url}</a>')

In [ ]:
IPython.display.Image(up_image_path)

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

down_image_path, down_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "down")

In [ ]:
IPython.display.HTML(f'<a href={down_url}>{down_url}</a>')

In [ ]:
IPython.display.Image(down_image_path)